In [ ]:
!pip install opencv-python-headless pillow matplotlib requests
!pip install diffusers transformers accelerate safetensors
!pip install torch torchvision               # GPU 付きランタイムなら不要な場合あり
!pip install pillow matplotlib requests

In [ ]:
# -----------------------------------------
# Imports & Setup
# -----------------------------------------
import cv2
import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image
from io import BytesIO

# -----------------------------------------
# 1. ライセンスフリー画像の取得
#    Wikimedia Commons より猫の写真を使用
# -----------------------------------------
url = 'https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg'
resp = requests.get(url)
img_pil = Image.open(BytesIO(resp.content)).convert('RGB')
img = np.array(img_pil)

# -----------------------------------------
# 2. マスクの作成
#    （ここでは例として、画像中の顔のあたりを長方形でマスク）
# -----------------------------------------
mask = np.zeros(img.shape[:2], dtype=np.uint8)
# マスク領域：縦 y=50〜300px，横 x=100〜350px
mask[50:300, 100:350] = 255

# -----------------------------------------
# 3. Inpainting（Telea アルゴリズム）
# -----------------------------------------
# OpenCV は BGR 形式なので変換
img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
inpainted_bgr = cv2.inpaint(img_bgr, mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
# 再度 RGB に戻す
inpainted = cv2.cvtColor(inpainted_bgr, cv2.COLOR_BGR2RGB)

# -----------------------------------------
# 4. 結果の表示
#    各プロットは個別の figure で出力
# -----------------------------------------
# 元画像
plt.figure()
plt.imshow(img)
plt.axis('off')
plt.title('Original Image')
plt.show()

# マスク（白：消去対象領域）
plt.figure()
plt.imshow(mask, cmap='gray')
plt.axis('off')
plt.title('Mask')
plt.show()

# Inpainting 後の画像
plt.figure()
plt.imshow(inpainted)
plt.axis('off')
plt.title('Inpainted Image')
plt.show()

In [ ]:
# -----------------------------------------
# Imports & Setup
# -----------------------------------------
import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image, ImageDraw
import requests
from io import BytesIO
import matplotlib.pyplot as plt

# 再現性のため乱数固定
torch.manual_seed(0)

# 使用デバイス
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------------------
# 1. ライセンスフリー画像の取得
# -----------------------------------------
url = "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"
resp = requests.get(url)
orig = Image.open(BytesIO(resp.content)).convert("RGB")

# -----------------------------------------
# 2. マスク画像の作成（白＝消去領域、黒＝保持領域）
#    例：猫の顔あたりを四角で消す
# -----------------------------------------
mask = Image.new("L", orig.size, 0)
draw = ImageDraw.Draw(mask)
# マスク領域：左上(100,50)、右下(350,300)
draw.rectangle([100, 50, 350, 300], fill=255)

# -----------------------------------------
# 3. Stable Diffusion Inpainting パイプラインのロード
# -----------------------------------------
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16
).to(device)

# optional: 安全フィルタリングをオフにしたい場合
pipe.safety_checker = lambda images, **kwargs: (images, False)

# -----------------------------------------
# 4. Inpainting 実行
# -----------------------------------------
# Generator を渡して再現性を保つ
generator = torch.Generator(device=device).manual_seed(0)

# 空のプロンプトでマスクした領域だけ生成
out = pipe(
    prompt="",
    image=orig,
    mask_image=mask,
    guidance_scale=7.5,
    generator=generator,
).images[0]

# -----------------------------------------
# 5. 結果の表示（別々の figure で）
# -----------------------------------------
for title, img in [
    ("Original Image", orig),
    ("Mask (white = erased)", mask),
    ("Inpainted Result", out)
]:
    plt.figure()
    if img.mode == "L":
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(img)
    plt.axis("off")
    plt.title(title)
    plt.show()